# 11 — Metadata Filtering RAG

> **Tier 2 | Retrieval Quality**

## The Problem

Pure vector search retrieves the semantically closest chunks globally —
it cannot restrict results to a specific document section, page range,
or topic category. When users ask scoped questions the wrong chunks
appear at the top:

```
Q: "What does the introduction say about climate sensitivity?"

Vector search (no filter):
  Rank 1 — Chapter 7, page 42 (score 0.91)
  Rank 2 — Appendix B, page 89 (score 0.89)
  Rank 3 — Introduction, page 2  (score 0.87)  ← correct but ranked 3rd

Filtered search (section='introduction'):
  Rank 1 — Introduction, page 2  (score 0.87)  ← only candidate from section
  Rank 2 — Introduction, page 3  (score 0.84)
```

## Metadata Filtering Solution

Tag every chunk with **structured payload** at index time, then apply
a `Filter` object at query time. Qdrant evaluates the filter **before**
the vector search, so it narrows the candidate pool cheaply:

```
Index time:
  chunk → embed → Qdrant point with payload:
    { page_num: 3, section: 'introduction', topic: 'temperature',
      char_count: 487, word_count: 76, has_numbers: true }

Query time:
  filter = Filter(must=[page_num in [1,10], topic=='temperature'])
  results = qdrant.query_points(query=embed(q), filter=filter, limit=5)
```

## Metadata Fields

| Field | Type | Example | Notes |
|-------|------|---------|-------|
| `page_num` | int | `3` | PDF page number |
| `char_count` | int | `487` | Chunk character length |
| `word_count` | int | `76` | Approximate word count |
| `has_numbers` | bool | `true` | Contains numeric data |
| `section` | str | `'introduction'` | Detected heading category |
| `topic` | str | `'temperature'` | Keyword-classified topic |
| `source` | str | `'climate.pdf'` | Origin file |


## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — rich payload"]
        PDF(["📄 climate.pdf"])
        PDF --> PAGES["Extract text\nper page"]
        PAGES --> SPLIT["Fixed-size chunks\n~500 chars"]
        SPLIT --> META["detect_section()\nclassify_topic()"]
        META --> TAG["Tag each chunk:\npage_num, section, topic,\nchar_count, has_numbers"]
        TAG --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\n+ payload index")]
    end

    subgraph FILTER ["🟡  FILTER STRATEGIES"]
        direction LR
        F1["Exact match\nsection='introduction'"]
        F2["Range\npage_num 1-20"]
        F3["Multi-condition\ntopic='temperature'\nAND has_numbers=true"]
        F4["Dynamic\nLLM extracts filter\nfrom query text"]
    end

    subgraph QUERY ["⚡  FILTERED RETRIEVAL"]
        Q(["❓ Query"])
        Q --> QEMB["embed(query)"]
        FILTER --> FOBJ["Filter object"]
        QEMB --> VS["query_points\n(query + filter)"]   
        FOBJ --> VS
        QDRANT --> VS
        VS --> HITS(["Top-K within\nfiltered subset"])
    end

    subgraph GEN ["🟠  GENERATION"]
        HITS --> LLM["Strands Agent\nClaude Sonnet 4.6"]
        LLM --> ANS(["✅ Answer"])
    end

    style INDEX   fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style FILTER  fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style QUERY   fill:#dcfce7,stroke:#16a34a,color:#14532d
    style GEN     fill:#ffedd5,stroke:#f97316,color:#7c2d12
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — rich payload"]
        PDF(["📄 climate.pdf"])
        PDF --> PAGES["Extract text\nper page"]
        PAGES --> SPLIT["Fixed-size chunks\n~500 chars"]
        SPLIT --> META["detect_section()\nclassify_topic()"]
        META --> TAG["Tag each chunk:\npage_num, section, topic,\nchar_count, has_numbers"]
        TAG --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\n+ payload index")]
    end

    subgraph FILTER ["🟡  FILTER STRATEGIES"]
        direction LR
        F1["Exact match\nsection='introduction'"]
        F2["Range\npage_num 1-20"]
        F3["Multi-condition\ntopic='temperature'\nAND has_numbers=true"]
        F4["Dynamic\nLLM extracts filter\nfrom query text"]
    end

    subgraph QUERY ["⚡  FILTERED RETRIEVAL"]
        Q(["❓ Query"])
        Q --> QEMB["embed(query)"]
        FILTER --> FOBJ["Filter object"]
        QEMB --> VS["query_points\n(query + filter)"]   
        FOBJ --> VS
        QDRANT --> VS
        VS --> HITS(["Top-K within\nfiltered subset"])
    end

    subgraph GEN ["🟠  GENERATION"]
        HITS --> LLM["Strands Agent\nClaude Sonnet 4.6"]
        LLM --> ANS(["✅ Answer"])
    end

    style INDEX   fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style FILTER  fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style QUERY   fill:#dcfce7,stroke:#16a34a,color:#14532d
    style GEN     fill:#ffedd5,stroke:#f97316,color:#7c2d12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [1]:
import subprocess, sys
packages = ["boto3","qdrant-client","opensearch-py","requests-aws4auth","strands-agents","pypdf"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")

All packages ready.


In [2]:
import os, json, time, uuid, re
from typing import List, Dict, Optional, Any

import boto3, pypdf
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, MatchAny, Range
)

print("Imports OK")

Imports OK


## Step 2 — Configuration

In [3]:
# AWS / Bedrock
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

# Vector DB
QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

# Collection
COLLECTION_NAME = "metadata_filtering_11"
EMBEDDING_DIM   = 1024
TOP_K           = 5

# Chunking
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# PDF
PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

# Topic taxonomy — keyword buckets
TOPIC_KEYWORDS = {
    'temperature' : ['temperature','warming','heat','thermal','lapse','celsius','fahrenheit'],
    'ocean'       : ['ocean','sea','marine','coastal','salinity','current','tide'],
    'precipitation': ['rain','precipitation','snow','flood','drought','moisture','humidity'],
    'emissions'   : ['emission','carbon','greenhouse','co2','methane','fossil','aerosol'],
    'policy'      : ['policy','agreement','mitigation','adaptation','target','commitment'],
    'atmosphere'  : ['atmosphere','stratosphere','troposphere','ozone','pressure','wind'],
}

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF        : {PDF_PATH}")
print(f"PDF exists : {os.path.exists(PDF_PATH)}")
print(f"Topics     : {list(TOPIC_KEYWORDS.keys())}")

Collection : metadata_filtering_11
PDF        : C:\Users\Administrator\RAG\data\climate.pdf
PDF exists : True
Topics     : ['temperature', 'ocean', 'precipitation', 'emissions', 'policy', 'atmosphere']


## Step 3 — Vector Store (with metadata filter support)

In [4]:
from typing import List, Dict, Optional, Any

class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists')
        elif self._backend == 'opensearch':
            if force_recreate and self._os.indices.exists(index=self.name):
                self._os.indices.delete(index=self.name)
            if not self._os.indices.exists(index=self.name):
                self._os.indices.create(index=self.name, body={
                    'settings': {'index': {'knn': True}},
                    'mappings': {'properties': {
                        'text':      {'type': 'text'},
                        'metadata':  {'type': 'object'},
                        'embedding': {
                            'type': 'knn_vector', 'dimension': dim,
                            'method': {'name': 'hnsw', 'space_type': 'cosinesimil',
                                       'engine': 'faiss',
                                       'parameters': {'ef_construction': 512, 'm': 16}}
                        }}}})
                print(f'Created OpenSearch index "{self.name}"')

    def create_payload_indexes(self):
        if self._backend not in ('qdrant_cloud', 'qdrant_memory'):
            print('Payload indexes only supported on Qdrant backend')
            return
        index_defs = [
            ('metadata.page_num',   'integer'),
            ('metadata.char_count', 'integer'),
            ('metadata.word_count', 'integer'),
            ('metadata.has_numbers','bool'),
            ('metadata.section',    'keyword'),
            ('metadata.topic',      'keyword'),
        ]
        for field, schema in index_defs:
            self._qdrant.create_payload_index(
                collection_name=self.name,
                field_name=field,
                field_schema=schema
            )
        print(f'Payload indexes created: {[f for f, _ in index_defs]}')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata',{})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)
        elif self._backend == 'opensearch':
            for d in docs: self._os.index(index=self.name, body=d)
            time.sleep(1)
            return len(docs)

    def search(
        self,
        qvec: List[float],
        top_k: int = 5,
        query_filter: Optional[Any] = None
    ) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name,
                query=qvec,
                limit=top_k,
                query_filter=query_filter,
                with_payload=True
            )
            return [{'text': p.payload.get('text',''),
                     'metadata': p.payload.get('metadata',{}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]
        elif self._backend == 'opensearch':
            body = {'size': top_k,
                    'query': {'knn': {'embedding': {'vector': qvec, 'k': top_k}}},
                    '_source': {'excludes': ['embedding']}}
            resp = self._os.search(index=self.name, body=body)
            return [{'text': h['_source'].get('text',''),
                     'metadata': h['_source'].get('metadata',{}),
                     'score': h['_score'], 'id': h['_id']}
                    for h in resp['hits']['hits']]

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0
        return 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        elif self._backend == 'opensearch':
            self._os.indices.delete(index=self.name, ignore=[404])
        print(f'Deleted "{self.name}"')

print("VectorStore defined (filter-aware).")

VectorStore defined (filter-aware).


## Step 4 — Bedrock Helpers

In [5]:
from typing import List

bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def generate_answer(question: str, context_chunks: List[str]) -> str:
    context = '\n\n'.join(f'[Passage {i+1}]\n{c}' for i, c in enumerate(context_chunks))
    prompt  = (
        'Use ONLY the context below to answer. '
        "If the answer is not present say 'Not found in context.'\n\n"
        f'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'
    )
    return str(Agent(
        model=_model,
        system_prompt='You are a precise Q&A assistant. Answer only from the provided context.'
    )(prompt))

test_emb = embed_text("metadata filtering retrieval")
print(f"Embedding OK — dim={len(test_emb)}")
print("BedrockModel ready.")

Embedding OK — dim=1024
BedrockModel ready.


## Step 5 — Connect, Create Collection & Payload Indexes

In [6]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL,
    qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL,
    region=AWS_REGION
)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

# Create payload indexes for fast pre-filter evaluation
vs.create_payload_indexes()

Qdrant Cloud: https://2210ff1c-7c49-4fec-91f4-01662586299c.us-east-1-1.aws.cloud.qdrant.io
Backend: qdrant_cloud
Created "metadata_filtering_11" (dim=1024)
Payload indexes created: ['metadata.page_num', 'metadata.char_count', 'metadata.word_count', 'metadata.has_numbers', 'metadata.section', 'metadata.topic']


## Step 6 — Section Detection & Topic Classification

Two pure-Python helpers that run at chunk-creation time:

- `detect_section(text)` — infers section category from text patterns
- `classify_topic(text)` — keyword voting across topic buckets


In [7]:
from typing import Dict, List

# Section patterns (checked in priority order)
SECTION_PATTERNS = [
    (r'(?i)\bintroduction\b',                    'introduction'),
    (r'(?i)\bconclusion\b|\bsummary\b',          'conclusion'),
    (r'(?i)\bmethod(?:s|ology)?\b',               'methods'),
    (r'(?i)\bresult(?:s)?\b|\bfinding(?:s)?\b',  'results'),
    (r'(?i)\bdiscussion\b',                       'discussion'),
    (r'(?i)\breference(?:s)?\b|\bbiblio\b',      'references'),
    (r'(?i)\bappendix\b|\bannex\b',              'appendix'),
    (r'(?i)\babstract\b|\bexecutive\s+summary\b','abstract'),
]

def detect_section(text: str) -> str:
    sample = text[:300]
    for pattern, label in SECTION_PATTERNS:
        if re.search(pattern, sample):
            return label
    if re.search(r'^[A-Z][A-Z\s]{8,50}$', sample.split('\n')[0].strip()):
        return 'header'
    if re.search(r'^\d+(\.\d+)*\s+\w', sample.strip()):
        return 'numbered'
    return 'body'

def classify_topic(text: str) -> str:
    lower = text.lower()
    scores = {topic: 0 for topic in TOPIC_KEYWORDS}
    for topic, keywords in TOPIC_KEYWORDS.items():
        for kw in keywords:
            scores[topic] += lower.count(kw)
    best = max(scores, key=lambda t: scores[t])
    return best if scores[best] > 0 else 'general'

# Spot checks
samples = [
    "Introduction to climate science and global warming observations.",
    "Ocean temperatures have risen significantly, salinity changes observed in coastal areas.",
    "Carbon dioxide emissions from fossil fuels have increased greenhouse gas concentrations.",
    "The methodology involved analysis of satellite data and ground-based observations.",
]
print("{:<55} {:>12}  {:>12}".format("Text", "Section", "Topic"))
print("-" * 82)
for s in samples:
    print("{:<55} {:>12}  {:>12}".format(s[:54], detect_section(s), classify_topic(s)))

Text                                                         Section         Topic
----------------------------------------------------------------------------------
Introduction to climate science and global warming obs  introduction   temperature
Ocean temperatures have risen significantly, salinity           body         ocean
Carbon dioxide emissions from fossil fuels have increa          body     emissions
The methodology involved analysis of satellite data an       methods       general


## Step 7 — Load PDF, Extract Per-Page Text & Build Tagged Chunks

Key difference from earlier notebooks: we extract text **page by page** so
each chunk knows its `page_num`. The chunker tracks which page each character
falls on and carries that forward into the metadata payload.


In [8]:
from typing import List, Dict

def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)

reader = pypdf.PdfReader(PDF_PATH)
n_pages = len(reader.pages)
print(f"PDF    : {PDF_PATH}")
print(f"Pages  : {n_pages}")

# Build tagged chunks: each chunk knows its page_num
tagged_chunks: List[Dict] = []
for page_idx, page in enumerate(reader.pages):
    page_num  = page_idx + 1
    page_text = page.extract_text() or ''
    page_chunks = recursive_split(page_text, CHUNK_SIZE, CHUNK_OVERLAP)
    for chunk in page_chunks:
        tagged_chunks.append({
            'text'     : chunk,
            'page_num' : page_num,
            'char_count': len(chunk),
            'word_count': len(chunk.split()),
            'has_numbers': bool(re.search(r'\d+\.?\d*', chunk)),
            'section'  : detect_section(chunk),
            'topic'    : classify_topic(chunk),
        })

total = len(tagged_chunks)
print(f"Chunks : {total}  |  avg {sum(c["char_count"] for c in tagged_chunks)//total} chars")

# Distribution summary
sections = {}; topics = {}
for c in tagged_chunks:
    sections[c['section']] = sections.get(c['section'], 0) + 1
    topics[c['topic']]     = topics.get(c['topic'], 0)     + 1

print("\nSection distribution:")
for k, v in sorted(sections.items(), key=lambda x: -x[1]):
    print(f"  {k:<18} {v:>4} chunks")
print("\nTopic distribution:")
for k, v in sorted(topics.items(), key=lambda x: -x[1]):
    print(f"  {k:<18} {v:>4} chunks")

PDF    : C:\Users\Administrator\RAG\data\climate.pdf
Pages  : 13
Chunks : 66  |  avg 423 chars

Section distribution:
  body                 51 chunks
  methods              14 chunks
  introduction          1 chunks

Topic distribution:
  general              32 chunks
  ocean                14 chunks
  temperature           8 chunks
  atmosphere            7 chunks
  precipitation         4 chunks
  policy                1 chunks


## Step 8 — Embed & Index with Rich Metadata Payload

In [9]:
from typing import List, Dict

print(f"Embedding {len(tagged_chunks)} chunks...")
t0   = time.time()
texts = [c['text'] for c in tagged_chunks]
embs  = embed_batch(texts, label='[index]')

docs = [
    {
        'text'     : tagged_chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'chunk_idx'  : i,
            'page_num'   : tagged_chunks[i]['page_num'],
            'char_count' : tagged_chunks[i]['char_count'],
            'word_count' : tagged_chunks[i]['word_count'],
            'has_numbers': tagged_chunks[i]['has_numbers'],
            'section'    : tagged_chunks[i]['section'],
            'topic'      : tagged_chunks[i]['topic'],
            'source'     : 'climate.pdf',
        }
    }
    for i in range(len(tagged_chunks))
]

vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")

# Inspect a sample point's payload
sample = docs[len(docs)//3]
print("\nSample payload:")
for k, v in sample['metadata'].items():
    print(f'  {k:<14}: {v}')

Embedding 66 chunks...
  [index] 20/66
  [index] 40/66
  [index] 60/66
Indexed 66 vectors in 9.9s

Sample payload:
  chunk_idx     : 22
  page_num      : 6
  char_count    : 498
  word_count    : 76
  has_numbers   : True
  section       : body
  topic         : atmosphere
  source        : climate.pdf


## Step 9 — Strategy A: Exact-Match Filter

Filter by a single categorical field (`section` or `topic`) before vector search.
Only chunks matching the field value participate in the ranking.


In [10]:
from typing import List, Dict

def rag_filtered(
    question: str,
    query_filter=None,
    label: str = 'filtered',
    verbose: bool = True
) -> Dict:
    t0   = time.time()
    qvec = embed_text(question)
    hits = vs.search(qvec, top_k=TOP_K, query_filter=query_filter)
    passages = [h['text'] for h in hits]
    answer   = generate_answer(question, passages) if passages else 'No results after filtering.'
    latency  = (time.time() - t0) * 1000
    if verbose:
        print(f'\n[{label}]  Q: {question}')
        print(f'  Hits: {len(hits)}  Latency: {latency:.0f}ms')
        for h in hits:
            m = h['metadata']
            print(f'  [p{m.get("page_num","?"):>3}] sec={m.get("section","?"):<13}'
                  f' topic={m.get("topic","?"):<13} score={h["score"]:.4f}')
        print(f'  Answer: {answer[:200]}')
        print('-' * 70)
    return {'question': question, 'answer': answer, 'hits': hits,
            'n_hits': len(hits), 'latency_ms': latency, 'label': label}

# Demo: exact match on section='body' vs section='introduction'
test_q = "What is the main focus of this document?"

# No filter baseline
rag_filtered(test_q, query_filter=None, label='no_filter')

# Filter: introduction section only
f_intro = Filter(must=[FieldCondition(
    key='metadata.section', match=MatchValue(value='introduction'))])
rag_filtered(test_q, query_filter=f_intro, label='section=introduction')

# Filter: emissions topic only
f_emis = Filter(must=[FieldCondition(
    key='metadata.topic', match=MatchValue(value='emissions'))])
rag_filtered("What causes greenhouse gas increases?",
             query_filter=f_emis, label='topic=emissions')

Based on the provided context, the main focus of this document is **weather forecasting** — specifically covering different **methods, techniques, procedures, and tools** used in weather forecasting, as well as **different types of weather forecasts** (such as nowcasting). The document also touches on the role of weather forecasting in various sectors (aviation, agriculture, disaster management, etc.) and the importance of providing **pre-warnings for extreme weather events** like cyclones, floods, and thunderstorms to prevent loss of life and property.
[no_filter]  Q: What is the main focus of this document?
  Hits: 5  Latency: 3146ms
  [p 11] sec=body          topic=temperature   score=0.0846
  [p 11] sec=body          topic=ocean         score=0.0844
  [p  9] sec=body          topic=general       score=0.0753
  [p  2] sec=methods       topic=general       score=0.0682
  [p  9] sec=body          topic=precipitation score=0.0638
  Answer: Based on the provided context, the main focus 

{'question': 'What causes greenhouse gas increases?',
 'answer': 'No results after filtering.',
 'hits': [],
 'n_hits': 0,
 'latency_ms': 124.16791915893555,
 'label': 'topic=emissions'}

## Step 10 — Strategy B: Range Filter

Filter chunks by numeric fields: `page_num`, `char_count`, or `word_count`.
Useful for 'what does the first half of the document say' type queries.


In [11]:
from typing import List, Dict

n_pages_total = len(reader.pages)
mid_page      = n_pages_total // 2
print(f"Total pages: {n_pages_total}  |  Midpoint: {mid_page}")

# Early pages (first quarter)
f_early = Filter(must=[FieldCondition(
    key='metadata.page_num',
    range=Range(gte=1, lte=max(1, n_pages_total // 4)))])

# Late pages (last quarter)
f_late = Filter(must=[FieldCondition(
    key='metadata.page_num',
    range=Range(gte=n_pages_total - n_pages_total // 4, lte=n_pages_total))])

# Dense chunks only (many words — likely prose, not headers)
f_dense = Filter(must=[FieldCondition(
    key='metadata.word_count',
    range=Range(gte=60))])

demo_q = "What are the key observations and findings?"

r_early = rag_filtered(demo_q, query_filter=f_early,
                       label=f'pages 1-{n_pages_total//4}')
r_late  = rag_filtered(demo_q, query_filter=f_late,
                       label=f'pages {n_pages_total - n_pages_total//4}-{n_pages_total}')
r_dense = rag_filtered(demo_q, query_filter=f_dense, label='word_count>=60')

Total pages: 13  |  Midpoint: 6
## Key Observations and Findings on Weather Forecasting

Based on the provided context, the following key observations and findings can be identified:

### Ancient/Traditional Methods
- Early weather forecasting was done in a **crude manner** by observing:
  - Sky colour
  - Wind direction
  - Cloud colour and cloud cover
  - Lightning and thunder
  - **Behaviour of animals and birds** as indicators of weather change
  - **Folklores** were also used as forecasting methods

### Importance of Weather Forecasting
- Weather forecasting **impacts all walks of life**, including:
  - **Outdoor games and tournaments** — success depends on favourable weather conditions
  - **Farming activities** — from sowing to harvesting, including:
    - Applying fertilizer
    - Irrigation needs
    - Crop harvesting
  - **Storage and transportation of crops**

### Historical Context
- Weather forecasting was prevalent **even before modern tools and satellite technology**
- I

## Step 11 — Strategy C: Multi-Condition Filter

Combine conditions with `must` (AND) and `should` (OR):

- `must` = ALL conditions must hold (logical AND)
- `should` = AT LEAST ONE condition must hold (logical OR)
- `must_not` = NONE of the conditions must hold (logical NOT)


In [12]:
from typing import List, Dict
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny, Range

# AND: topic=temperature AND has_numbers=True
f_temp_numbers = Filter(must=[
    FieldCondition(key='metadata.topic',       match=MatchValue(value='temperature')),
    FieldCondition(key='metadata.has_numbers', match=MatchValue(value=True)),
])

# OR: topic in [emissions, atmosphere]  (single field, multiple values)
f_air_quality = Filter(must=[
    FieldCondition(key='metadata.topic',
                   match=MatchAny(any=['emissions', 'atmosphere'])),
])

# AND + range: early pages AND data-rich chunks
f_early_dense = Filter(must=[
    FieldCondition(key='metadata.page_num',   range=Range(gte=1, lte=10)),
    FieldCondition(key='metadata.word_count', range=Range(gte=50)),
])

# NOT: exclude references/appendix sections
from qdrant_client.models import Filter as QFilter
f_no_refs = QFilter(must_not=[
    FieldCondition(key='metadata.section',
                   match=MatchAny(any=['references', 'appendix'])),
])

q_temp = "What quantitative evidence exists for temperature changes?"
q_air  = "How do atmospheric emissions affect climate?"

rag_filtered(q_temp, query_filter=f_temp_numbers,  label='temp+has_numbers')
rag_filtered(q_air,  query_filter=f_air_quality,   label='emissions OR atmosphere')
rag_filtered(q_temp, query_filter=f_early_dense,   label='pages 1-10 AND dense')
rag_filtered(q_air,  query_filter=f_no_refs,       label='NOT references/appendix')

Based on the provided context, the only quantitative temperature reference is the example of **37°C** mentioned in Passage 2, where it states: *"if it is sunny today with the temperature of 37°C, then tomorrow also, it would be the same"* — used to illustrate the **persistence method** of weather prediction. However, this is not evidence of temperature *changes*, but rather an example of temperature *remaining the same*.

There is **no quantitative evidence for temperature changes** provided in the given context.
[temp+has_numbers]  Q: What quantitative evidence exists for temperature changes?
  Hits: 3  Latency: 3163ms
  [p  4] sec=body          topic=temperature   score=0.2066
  [p 10] sec=methods       topic=temperature   score=0.1771
  [p  5] sec=body          topic=temperature   score=0.1592
  Answer: Based on the provided context, the only quantitative temperature reference is the example of **37°C** mentioned in Passage 2, where it states: *"if it is sunny today with the tempera

{'question': 'How do atmospheric emissions affect climate?',
 'answer': '**Not found in context.**\n\nThe provided passages discuss weather analysis, forecasting methods, atmospheric conditions, satellite sensors, and data compilation, but do not contain information specifically about how atmospheric emissions affect climate.\n',
 'hits': [{'text': 'WEATHER ANALYSIS AND WEATHERFORECASTING.\nINTRODUCTION: -\nIn this course so far, you have studied about the atmospheric processes in general as well\nas few extreme weather events. You are also well acquainted with classification of climates\ngiven by some of the leading scholars. You have also learned about the processes that have\nbrought about a change in climate. In this module, you will study about the processes of',
   'metadata': {'chunk_idx': 1,
    'page_num': 2,
    'char_count': 419,
    'word_count': 68,
    'has_numbers': False,
    'section': 'introduction',
    'topic': 'general',
    'source': 'climate.pdf'},
   'score': 0.

## Step 12 — Strategy D: Dynamic Filter Extraction

For production systems, users rarely specify filters explicitly — they just
ask natural-language questions. Ask Claude to extract structured filter
parameters from the query text, then build the `Filter` object automatically.

```
Input  : "What does the introduction say about ocean warming?"
Output : {section: 'introduction', topic: 'ocean', page_gte: null, page_lte: null}
```


In [13]:
from typing import List, Dict, Optional

FILTER_EXTRACT_PROMPT = '''Extract metadata filter parameters from the question below.
Return ONLY a valid JSON object with these exact keys (use null for unspecified):
  page_gte   : int or null   (minimum page number the user cares about)
  page_lte   : int or null   (maximum page number the user cares about)
  section    : str or null   (one of: introduction, conclusion, methods, results, discussion, references, appendix, abstract, body)
  topic      : str or null   (one of: temperature, ocean, precipitation, emissions, policy, atmosphere, general)
  has_numbers: bool or null  (true if user asks for quantitative/numeric data)

Question: {question}

JSON:'''

def extract_filters(question: str) -> Dict:
    prompt = FILTER_EXTRACT_PROMPT.format(question=question)
    raw = str(Agent(
        model=_model,
        system_prompt='You extract JSON metadata. Return only valid JSON, no extra text.'
    )(prompt)).strip()
    # Parse JSON from response
    match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {'page_gte': None, 'page_lte': None, 'section': None,
            'topic': None, 'has_numbers': None}

def build_filter(params: Dict) -> Optional[Filter]:
    must = []
    if params.get('page_gte') or params.get('page_lte'):
        must.append(FieldCondition(
            key='metadata.page_num',
            range=Range(
                gte=params.get('page_gte') or 1,
                lte=params.get('page_lte') or 9999
            )))
    if params.get('section'):
        must.append(FieldCondition(
            key='metadata.section',
            match=MatchValue(value=params['section'])))
    if params.get('topic'):
        must.append(FieldCondition(
            key='metadata.topic',
            match=MatchValue(value=params['topic'])))
    if params.get('has_numbers') is True:
        must.append(FieldCondition(
            key='metadata.has_numbers',
            match=MatchValue(value=True)))
    return Filter(must=must) if must else None

def rag_dynamic(
    question: str,
    verbose: bool = True
) -> Dict:
    params = extract_filters(question)
    flt    = build_filter(params)
    if verbose:
        print(f'\nQ: {question}')
        print(f'Extracted filters: {params}')
        print(f'Filter built: {flt is not None}')
    return rag_filtered(question, query_filter=flt,
                        label='dynamic', verbose=verbose)

dynamic_qs = [
    "What does the introduction explain about climate observations?",
    "Give me quantitative data about ocean temperature changes.",
    "What does the conclusion say about future policy recommendations?",
]
for q in dynamic_qs:
    rag_dynamic(q)

```json
{
  "page_gte": null,
  "page_lte": null,
  "section": "introduction",
  "topic": "general",
  "has_numbers": null
}
```
Q: What does the introduction explain about climate observations?
Extracted filters: {'page_gte': None, 'page_lte': None, 'section': 'introduction', 'topic': 'general', 'has_numbers': None}
Filter built: True
Based on the provided context, the introduction does **not explicitly explain about climate observations**. It only mentions that students have studied atmospheric processes, extreme weather events, climate classification by leading scholars, and processes that have brought about climate change.

**Not found in context.**
[dynamic]  Q: What does the introduction explain about climate observations?
  Hits: 1  Latency: 2156ms
  [p  2] sec=introduction  topic=general       score=0.4129
  Answer: Based on the provided context, the introduction does **not explicitly explain about climate observations**. It only mentions that students have studied atmospheric 

## Step 13 — Filtered vs Unfiltered Comparison

Run the same questions with no filter, static filter, and dynamic filter.
Compare: number of hits, average page spread, and keyword hit rate.


In [14]:
from typing import List, Dict

eval_pairs = [
    {
        "question"   : "What are the observed changes in atmospheric temperature?",
        "keywords"   : ["temperature","change","observed","atmospheric","warming"],
        "static_filter": Filter(must=[FieldCondition(
            key='metadata.topic', match=MatchValue(value='temperature'))]),
        "filter_label" : "topic=temperature",
    },
    {
        "question"   : "What are the conclusions about emissions reductions?",
        "keywords"   : ["conclusion","emission","reduction","target","policy"],
        "static_filter": Filter(must=[FieldCondition(
            key='metadata.section', match=MatchValue(value='conclusion'))]),
        "filter_label" : "section=conclusion",
    },
    {
        "question"   : "What numerical data supports ocean warming trends?",
        "keywords"   : ["ocean","warming","data","degree","temperature"],
        "static_filter": Filter(must=[
            FieldCondition(key='metadata.topic', match=MatchValue(value='ocean')),
            FieldCondition(key='metadata.has_numbers', match=MatchValue(value=True)),
        ]),
        "filter_label" : "ocean+numbers",
    },
]

print('{:<46} {:>9}  {:>9}  {:>9}  {:>9}'.format(
    'Question', 'NoFilter', 'Static', 'Dynamic', 'Pg spread'))
print('-' * 90)

for ep in eval_pairs:
    q   = ep['question']
    kws = ep['keywords']
    n   = len(kws)

    r_none    = rag_filtered(q, query_filter=None,           label='none',    verbose=False)
    r_static  = rag_filtered(q, query_filter=ep['static_filter'],
                             label=ep['filter_label'],       verbose=False)
    r_dynamic = rag_dynamic(q, verbose=False)

    def kw_hits(r):
        return sum(1 for kw in kws if kw in r['answer'].lower())

    def page_spread(r):
        pages = [h['metadata'].get('page_num', 0) for h in r['hits']]
        return max(pages) - min(pages) if len(pages) > 1 else 0

    print('{:<46} {:>4}/{} hits  {:>4}/{} hits  {:>4}/{} hits  {:>5}'.format(
        q[:45],
        kw_hits(r_none),    n,
        kw_hits(r_static),  n,
        kw_hits(r_dynamic), n,
        page_spread(r_none)))

print()
print('Format: keyword_hits/total  Page spread = max_page - min_page across top-K hits')

Question                                        NoFilter     Static    Dynamic  Pg spread
------------------------------------------------------------------------------------------
**Not found in context.**

The provided passages discuss weather forecasting methods and general atmospheric conditions, but do not contain specific information about observed changes in atmospheric temperature.**Not found in context.**

The provided passages do not contain specific information about observed changes in atmospheric temperature. They discuss weather instruments, forecasting methods, and atmospheric measurements, but do not address observed changes or trends in atmospheric temperature.```json
{
  "page_gte": null,
  "page_lte": null,
  "section": null,
  "topic": "temperature",
  "has_numbers": null
}
```**Not found in context.**

The passages provided discuss weather forecasting methods, instruments used to measure atmospheric conditions, and weather data collection systems, but do not specif

## Step 14 — Summary

| Component | Implementation |
|-----------|---------------|
| PDF | `pypdf.PdfReader` — page-by-page extraction preserving `page_num` |
| Metadata fields | `page_num`, `char_count`, `word_count`, `has_numbers`, `section`, `topic` |
| Section detection | Regex patterns on first 300 chars of each chunk |
| Topic classification | Keyword vote across 6 topic buckets |
| Payload indexes | `create_payload_index()` per field — accelerates pre-filter |
| Filter A | Exact match — `FieldCondition` + `MatchValue` |
| Filter B | Range — `FieldCondition` + `Range(gte, lte)` |
| Filter C | Multi-condition — `Filter(must=[...])` AND / `must_not` NOT |
| Filter D | Dynamic — LLM extracts JSON params → `build_filter()` |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |
| LLM | AWS Strands Agent + Claude Sonnet 4.6 |

## Filter strategy comparison

| Strategy | Setup | Best for | Qdrant API |
|----------|-------|----------|------------|
| Exact match | Low | Category-scoped Q&A | `MatchValue` |
| Range | Low | Page/length scoping | `Range(gte, lte)` |
| Multi-condition | Medium | Complex scoping | `must` / `should` / `must_not` |
| Dynamic | LLM call | Natural-language filters | JSON → `build_filter()` |

**When to use metadata filtering:**
- Documents with clear structure (sections, chapters, dates)
- Multi-document corpora — filter by `doc_id` before vector search
- Date-scoped questions (`published_after`, `year`)
- Compliance / auditing — restrict to authoritative sections only

### Next: **12 — Maximal Marginal Relevance (MMR)**


In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print("\nDone. Give the go-ahead for notebook 12.")